In [1]:
from neo4j import GraphDatabase, RoutingControl, basic_auth

URL = "neo4j://localhost:7687"
AUTH =("neo4j", "badainamko")

In [2]:
def add_friend(driver, name, friend_name): 
    driver.execute_query( 
        "MERGE (a:Person {name: $name}) " 
        "MERGE (friend:Person {name: $friend_name}) " 
        "MERGE (a)-[:KNOWS]->(friend)", 
        name=name,  friend_name=friend_name,  database_="neo4j", 
    ) 

In [4]:
def  print_friends(driver,  name): 
    records, _, _ = driver.execute_query( 
        "MATCH (a:Person)-[:KNOWS]->(friend) WHERE a.name = $name " 
        "RETURN friend.name ORDER BY friend.name", 
        name=name, database_="neo4j", routing_=RoutingControl.READ, 
    ) 
    for record in records: 
        print(record["friend.name"]) 
with GraphDatabase.driver(URL, auth=AUTH, ) as driver: 
    add_friend(driver, "Arthur", "Guinevere") 
    add_friend(driver, "Arthur", "Lancelot") 
    add_friend(driver, "Arthur", "Merlin") 
    print_friends(driver, "Arthur") 

Guinevere
Lancelot
Merlin


## 1.2 Pertanyaan Pendahuluan (Checkpoint)

### 1. Apa nama database yang digunakan pada tutorial ini?

Database yang digunakan bernama **`neo4j`**, yaitu database default Neo4j. Nama ini ditentukan melalui parameter `database_="neo4j"` pada `driver.execute_query(...)` di fungsi `add_friend` dan `print_friends`. Selain itu, setiap instance Neo4j memiliki database sistem bernama `system` yang digunakan untuk administrasi.

### 2. Akses lain (sintaks DDL/DML Cypher) untuk mengakses database lain dalam 1 instance

| Cara | Contoh | Keterangan |
|---|---|---|
| Klausa `USE` | `USE namadb MATCH (n) RETURN n` | Pada Neo4j Browser/Query juga dapat memakai perintah `:use namadb` |
| Python driver (session) | `driver.session(database="namadb")` | Menentukan database saat membuat session |
| Python driver (execute_query) | `driver.execute_query(..., database_="namadb")` | Menentukan database per query |
| Perintah administrasi | `SHOW DATABASES`, `CREATE DATABASE namadb`, `START DATABASE namadb`, `STOP DATABASE namadb`, `DROP DATABASE namadb` | Dijalankan pada database `system` |

### 3. Apa jenis graf pertemanan yang telah dibuat di sini?

Graf yang dibuat memiliki karakteristik berikut:

- **Berarah (directed)**: relasi `(a)-[:KNOWS]->(friend)` memiliki arah, sehingga "Arthur mengenal Guinevere" belum tentu berlaku sebaliknya.
- **Unipartit**: hanya terdapat satu jenis simpul, yaitu `Person`.
- **Satu jenis relasi**: `KNOWS`.
- **Tidak berbobot**: relasi tidak memiliki properti bobot.
- **Graf sederhana**: tidak ada relasi ganda, karena `MERGE` mencegah duplikasi simpul dan relasi.
- **Berbentuk bintang (star)**: Arthur menjadi hub dengan 3 relasi keluar ke Guinevere, Lancelot, dan Merlin, sehingga totalnya 4 simpul dan 3 relasi.